### Compas Analysis (individual assign 1)

#### Step 1. Loading data 


In this step, we load the COMPAS dataset from an online source using pandas.  
This dataset contains criminal defendant information and is commonly used for fairness analysis in machine learning.

We first import the necessary library, then read the dataset from a URL into a pandas DataFrame.  
Finally, we check the shape of the dataset to understand how many observations (rows) and variables (columns) are included.

In [1]:
import pandas as pd # Import pandas library for data manipulation and analysis
# Define the URL where the dataset is stored
# This dataset is from github/ProPublica 
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
# Read the CSV file from the URL into a pandas DataFrame
# The DataFrame 'raw_data' will store all the original data
raw_data = pd.read_csv(url)
# Print the shape of the dataset (rows, columns)
# This helps us understand the size of the dataset
print(raw_data.shape)

(7214, 53)


Observing the result above, we can know that this data has 7214 raws with 53 columns.

#### Step 2 Data Preview


In this step, we display the first 10 rows of the dataset using `df.head(10)` to gain an initial understanding of the data.

This allows us to:
- Inspect the structure and layout of the dataset
- Identify the available variables and their meanings (e.g., age, race, priors_count)
- Check whether the data has been loaded correctly
- Get a sense of the data types (categorical vs numerical)

Previewing the data is an important first step in any data analysis workflow, as it helps us understand what kind of preprocessing and feature engineering may be needed in the following steps.

In [3]:
# Show the first 10 rows of the dataset
raw_data.head(10)

,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0
5,7,marsha miles,marsha,miles,2013-11-30,Male,1971-08-22,44,25 - 45,Other,...,1,Low,2013-11-30,2013-11-30,2013-12-01,0,1,853,0,0
6,8,edward riddle,edward,riddle,2014-02-19,Male,1974-07-23,41,25 - 45,Caucasian,...,2,Low,2014-02-19,2014-03-31,2014-04-18,14,5,40,1,1
7,9,steven stewart,steven,stewart,2013-08-30,Male,1973-02-25,43,25 - 45,Other,...,3,Low,2013-08-30,2014-05-22,2014-06-03,3,0,265,0,0
8,10,elizabeth thieme,elizabeth,thieme,2014-03-16,Female,1976-06-03,39,25 - 45,Caucasian,...,1,Low,2014-03-16,2014-03-15,2014-03-18,0,2,747,0,0
9,13,bo bradac,bo,bradac,2013-11-04,Male,1994-06-10,21,Less than 25,Caucasian,...,5,Medium,2013-11-04,2015-01-06,2015-01-07,1,0,428,1,1


#### Step 3 Data Preprocessing and Feature Engineering



In this step, we clean and preprocess the dataset to prepare it for analysis and modeling.

First, we select only the relevant variables from the raw dataset to focus on meaningful features.  
Then, we apply several filtering conditions to remove invalid or irrelevant observations, such as missing values,and undefined categories.

Next, we convert variables into appropriate data types.

Finally, we create new derived variables (feature engineering), including categorical factors with specified reference levels. These transformations are important for statistical modeling, especially when interpreting regression coefficients.

Overall, this step ensures that the dataset is clean, consistent, and suitable for further analysis.

In [4]:
import pandas as pd
# Define groups of variables for later type conversion
# numeric_vars: variables that should remain numeric
# datetime_vars: variables that represent timestamps
numeric_vars = ["age", "priors_count", "days_b_screening_arrest", "decile_score"]
datetime_vars = ["c_jail_in", "c_jail_out"]

# Select only relevant columns from the raw dataset
# This helps reduce noise and focus on variables used in analysis
df = raw_data[
    [
        "age",
        "c_charge_degree",
        "race",
        "age_cat",
        "score_text",
        "sex",
        "priors_count",
        "days_b_screening_arrest",
        "decile_score",
        "is_recid",
        "two_year_recid",
        "c_jail_in",
        "c_jail_out",
    ]
].copy()

# -------------------------------
# Data Filtering 
# -------------------------------


# Keep only observations where screening arrest happens within [-30, 30] days
df = df[
    df["days_b_screening_arrest"].between(-30, 30)
]
# Remove rows with invalid recidivism values (-1 means missing/undefined)
df = df[df["is_recid"] != -1]
# Remove observations with charge degree "O" (unclassified/other)
df = df[df["c_charge_degree"] != "O"]
# Remove rows with missing COMPAS score
df = df[df["score_text"] != "N/A"]

# -------------------------------
# Type Conversion
# -------------------------------


# Convert datetime columns into proper datetime format
# errors="coerce" will turn invalid parsing into NaT (missing datetime)
for col in datetime_vars:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d %H:%M:%S", utc=True, errors="coerce")

# Convert all non-numeric and non-datetime columns into categorical variables
# This is important for modeling and memory efficiency
categorical_cols = [col for col in df.columns if col not in numeric_vars + datetime_vars]
for col in categorical_cols:
    df[col] = df[col].astype("category")

# -------------------------------
# Feature Engineering
# -------------------------------

# Create a categorical version of charge degree
df["crime_factor"] = df["c_charge_degree"].astype("category")

# Create age categories with a defined reference group ("25 - 45")
# The first category will act as baseline in modeling
df["age_factor"] = pd.Categorical(
    df["age_cat"],
    categories=["25 - 45", "Less than 25", "Greater than 45"],
    ordered=False
)
# Define race categories with "Caucasian" as the reference group (first position)
# We dynamically extract all unique race values from the dataset (excluding NaN),
# and then prepend "Caucasian" to ensure it is used as the baseline in modeling.

# This approach is more flexible than hardcoding all categories, because:
# - It automatically adapts to the dataset if new or unexpected race values appear
# - It avoids accidentally omitting valid categories

# If categories are manually hardcoded and do not include all possible values,
# any unmatched values will be converted to NaN when creating the categorical variable.
# This may silently lead to data loss or biased results if not handled properly.

df["race_factor"] = pd.Categorical(
    df["race"],
    categories=["Caucasian"] + [x for x in df["race"].dropna().unique() if x != "Caucasian"],
    ordered=False
)
df["race_factor"] = pd.Categorical(
    df["race"],
    categories=["Caucasian"] + [x for x in df["race"].dropna().unique() if x != "Caucasian"],
    ordered=False
)

# gender_factor: match Female / Male labels, with Male as reference
# assuming original sex values are Female and Male
df["gender_factor"] = pd.Categorical(
    df["sex"].replace({"Female": "Female", "Male": "Male"}),
    categories=["Male", "Female"],
    ordered=False
)

# Create binary score variable:
# Low -> LowScore
# Medium/High -> HighScore
df["score_factor"] = pd.Categorical
df["score_factor"] = pd.Categorical(
    # We use .ne("Low") to create a boolean mask where "Low" becomes False and others become True
    df["score_text"].ne("Low").map({False: "LowScore", True: "HighScore"}),
    categories=["LowScore", "HighScore"],
    ordered=False
)

# Number of rows
print(len(df))


6172



After applying the filtering conditions, the number of observations is reduced to 6,172 from the original dataset.

This reduction indicates that a portion of the data has been removed due to:
- Missing or undefined values (e.g., is_recid = -1, score_text = "N/A")
- Irrelevant or out-of-scope observations (e.g., c_charge_degree = "O")
- Observations outside the specified screening window (±30 days)

These filtering steps are necessary to ensure data quality and consistency. 

#### Step 4 Filtered Data Preview

After applying the filtering and preprocessing steps, we display the first few rows of the cleaned dataset using df.head().This step serves as a quick check before proceeding to further analysit.

In [5]:
df.head()

,age,c_charge_degree,race,age_cat,score_text,sex,priors_count,days_b_screening_arrest,decile_score,is_recid,two_year_recid,c_jail_in,c_jail_out,crime_factor,age_factor,race_factor,gender_factor,score_factor
0,69,F,Other,Greater than 45,Low,Male,0,-1.0,1,0,0,2013-08-13 06:03:42+00:00,2013-08-14 05:41:20+00:00,F,Greater than 45,Other,Male,LowScore
1,34,F,African-American,25 - 45,Low,Male,0,-1.0,3,1,1,2013-01-26 03:45:27+00:00,2013-02-05 05:36:53+00:00,F,25 - 45,African-American,Male,LowScore
2,24,F,African-American,Less than 25,Low,Male,4,-1.0,4,1,1,2013-04-13 04:58:34+00:00,2013-04-14 07:02:04+00:00,F,Less than 25,African-American,Male,LowScore
5,44,M,Other,25 - 45,Low,Male,0,0.0,1,0,0,2013-11-30 04:50:18+00:00,2013-12-01 12:28:56+00:00,M,25 - 45,Other,Male,LowScore
6,41,F,Caucasian,25 - 45,Medium,Male,14,-1.0,6,1,1,2014-02-18 05:08:24+00:00,2014-02-24 12:18:30+00:00,F,25 - 45,Caucasian,Male,HighScore


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6172 entries, 0 to 7213
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   age                      6172 non-null   int64              
 1   c_charge_degree          6172 non-null   category           
 2   race                     6172 non-null   category           
 3   age_cat                  6172 non-null   category           
 4   score_text               6172 non-null   category           
 5   sex                      6172 non-null   category           
 6   priors_count             6172 non-null   int64              
 7   days_b_screening_arrest  6172 non-null   float64            
 8   decile_score             6172 non-null   int64              
 9   is_recid                 6172 non-null   category           
 10  two_year_recid           6172 non-null   category           
 11  c_jail_in                6172 non-n


We use `df.info()` to quickly check the structure of the dataset.  
From the output, we can see that the data types have been correctly converted (e.g., categorical and datetime), and there are no missing values in the selected variables. This confirms that the dataset is clean and ready for further analysis.